In [ ]:
import pandas as pd
import sqlite3
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
csv_files = [
    ('customers.csv', 'customers'),
    ('orders.csv', 'orders'),
    ('sellers.csv', 'sellers'),
    ('products.csv', 'products'),
    ('geolocation.csv', 'geolocation'),
    ('payments.csv', 'payments'),
    ('order_items.csv', 'order_items')
]

folder_path = "/data"

# Creates a local database file — no server needed
conn = sqlite3.connect("ecommerce.db")

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    try:
        df = pd.read_csv(file_path)
        df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') 
                      for col in df.columns]
        
        # One line loads entire CSV into database
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"✅ {table_name} loaded — {len(df)} rows")
        
    except FileNotFoundError:
        print(f"❌ Not found: {file_path}")

conn.close()
print("\n✅ All done")

In [ ]:
conn = sqlite3.connect("ecommerce.db")
cur=conn.cursor()

1)List all unique cities where customers are located

In [ ]:
query="""select distinct customer_city from customers"""

df=pd.read_sql_query(query,conn)
df

2)Count the no. of orders placed in 2017

In [ ]:
query="""select count(order_id) from orders where strftime('%Y',order_purchase_timestamp)=2017"""


cur.execute(query)
data=cur.fetchall()

"total orders in 2017 are",data[0][0]

3)Find the total sales per category

In [ ]:
query="""select products.product_category,round(sum(payments.payment_value),2) as total_sales
from products join order_items
on products.product_id=order_items.product_id
join payments
on payments.order_id=order_items.order_id
group by product_category"""

df=pd.read_sql_query(query,conn)
df

4)Calculate the percentage of orders that were paid in installments

In [ ]:
query="""select 
round((sum(case when payment_installments> 1 then 1 else 0 end)*100/count(*)),2)
from payments"""

cur.execute(query)

data=cur.fetchall()

"percentage of orders pain in installments",data[0][0]

5)Count the number of customers from each state

In [ ]:
query="""select customer_state, count(customer_id) from customers group by customer_state"""

cur.execute(query)

data=cur.fetchall()

df=pd.DataFrame(data,columns=["State","No. of customers"])
df=df.sort_values(by="No. of customers", ascending=False)

plt.figure(figsize=(9,4))
plt.bar(df["State"],df["No. of customers"])
plt.xticks(rotation=90)
plt.xlabel("State")
plt.ylabel("No. of customers")
plt.tight_layout()
plt.show()

6)Calculate the no. of orders per month in 2018

In [ ]:
query="""select 
    case strftime('%m', order_purchase_timestamp)
        when '01' then 'January'
        when '02' then 'February'
        when '03' then 'March'
        when '04' then 'April'
        when '05' then 'May'
        when '06' then 'June'
        when '07' then 'July'
        when '08' then 'August'
        when '09' then 'September'
        when '10' then 'October'
        when '11' then 'November'
        when '12' then 'December'
    end as month_name,
    count(order_id) AS total_orders
from orders
where strftime('%Y', order_purchase_timestamp) = '2018'
group by strftime('%m', order_purchase_timestamp)
order by strftime('%m', order_purchase_timestamp);"""

cur.execute(query)

data=cur.fetchall()
df=pd.DataFrame(data, columns=["Months","No. of orders"])
df

plt.figure(figsize=(12,6))
plt.bar(df['Months'],df['No. of orders'])
plt.xticks(rotation=90)
plt.xlabel("Months")
plt.ylabel("No. of orders")
plt.tight_layout()
plt.show()

7)Find the average no. of products per order, grouped by customer city

In [ ]:
query="""with count_per_order as(select orders.order_id, orders.customer_id, count(order_items.order_id) as oc
from orders
join order_items
on orders.order_id = order_items.order_id
group by orders.order_id, orders.customer_id)

select customers.customer_city, round(avg(count_per_order.oc),2) as avg_no_of_products_per_order from count_per_order
join customers 
on customers.customer_id = count_per_order.customer_id
group by customers.customer_city 
order by avg_no_of_products_per_order desc"""

df=pd.read_sql_query(query,conn)
df

8)Calculate the percentage of total revenue contributed by each product category

In [ ]:
query="""select upper(products.product_category) as category,
round((sum(payments.payment_value)/(select sum(payment_value) from payments))*100,2) as sales_percentage
from products join order_items
on products.product_id=order_items.product_id
join payments
on payments.order_id=order_items.order_id
group by category
order by sales_percentage desc"""

df=pd.read_sql_query(query,conn)
df

9)Identify the correlation between product price and the no of times a product has been purchased

In [ ]:
query="""select products.product_category as category,
count(order_items.product_id) as count, round(avg(order_items.price),2) as avg_price
from products join order_items
on products.product_id = order_items.product_id
group by category"""

df=pd.read_sql_query(query,conn)
df

In [ ]:
arr1=df["count"]
arr2=df["avg_price"]

a=np.corrcoef([arr1,arr2])
print("The correlation between product price and the no. of times a product has been purchased is", a[0][1])

10)Calculate the total revenue generated by a seller and rank them by revenue

In [ ]:
query="""select *, dense_rank() over(order by revenue desc)  as rnk from
(select order_items.seller_id as seller_id, round(sum(payments.payment_value),2) as revenue
from order_items join payments
on order_items.order_id = payments.order_id
group by seller_id ) as a"""

df=pd.read_sql_query(query,conn)
df

11)Calculate the moving average of order values for each customer over their order history

In [ ]:
query="""with order_payments as
(select orders.customer_id, orders.order_purchase_timestamp, payments.payment_value as payment
from orders join payments
on orders.order_id= payments.order_id)
select customer_id, order_purchase_timestamp, avg(payment) over(partition by customer_id 
order by order_purchase_timestamp rows between unbounded preceding and current row) as Moving_Average from order_payments"""

df=pd.read_sql_query(query,conn)
df

12)Calculate the cumulative sales per month for each year

In [ ]:
query="""with sales as
(select strftime('%Y',orders.order_purchase_timestamp) as years, 
strftime('%M',orders.order_purchase_timestamp) as months,
round(sum(payments.payment_value),2) as payment
from orders join payments
on orders.order_id = payments.order_id
group by years,months
order by years,months)
select years, months,payment, sum(payment) over(order by years,months) as cumulative_sales from sales """

df=pd.read_sql_query(query,conn)
df

13)Calculate the year-over-year growth rate of total sales

In [ ]:
query="""with sales as 
(select strftime('%Y',orders.order_purchase_timestamp) as years,
round(sum(payments.payment_value),2) as payment
from orders join payments
on orders.order_id = payments.order_id
group by years
order by years)

select years,round((payment- lag(payment,1) over(order by years))/lag(payment,1) over(order by years)*100,2)
as growth_rate from sales"""

df=pd.read_sql_query(query,conn)
df

14)Calculate the retention rate of customers,defined as the % of customers who make another purchase within 6 months

In [ ]:
query="""with first_purchase as
(select customers.customer_id, min(orders.order_purchase_timestamp) as first_purchase_date
from customers join orders
on customers.customer_id = orders.customer_id
group by customers.customer_id),


returning_customer as
(select first_purchase.customer_id, min(orders.order_purchase_timestamp) as next_purchase
from first_purchase join orders
on first_purchase.customer_id = orders.customer_id
where orders.order_purchase_timestamp > first_purchase_date
and orders.order_purchase_timestamp <= date(first_purchase_date, '+6 months')
group by first_purchase.customer_id
)

select count(distinct returning_customer.customer_id) *100/ count(distinct first_purchase.customer_id) 
as retention_rate
from first_purchase  left join  returning_customer 
on first_purchase.customer_id= returning_customer.customer_id
"""

df=pd.read_sql_query(query,conn)
df

15)Identify the top 3 customers who spent the most money in each year

In [ ]:
query="""with yearly_spending as
(select  STRFTIME('%Y',o.order_purchase_timestamp) as years,o.customer_id,round(sum(p.payment_value),2) as total_spent
from payments as p join orders as o
on p.order_id = o.order_id
group by years,o.customer_id), 

ranked_customers as
(select years, customer_id, total_spent, rank() over(partition by years order by total_spent desc) as rnk
from yearly_spending)

select years, customer_id, total_spent, rnk
from ranked_customers
where rnk<=3
order by  years,rnk """

df=pd.read_sql_query(query,conn)
df

16)Find the average delivery time per state

In [ ]:
query="""select customers.customer_state, 
round(avg(julianday(orders.order_delivered_customer_date)-julianday(orders.order_purchase_timestamp)),2) as avg_time
from customers join orders
on customers.customer_id = orders.customer_id
where orders.order_status = 'delivered'
and orders.order_delivered_customer_date IS NOT NULL
group by customers.customer_state
order by avg_time"""

cur.execute(query)
data=cur.fetchall()
df=pd.DataFrame(data,columns=["State","Avg. delivery time"])
df

17)Which seller has the highest cancellation rate

In [ ]:
query="""with seller_orders as
(select oi.seller_id, o.order_status, count(distinct o.order_id) as total_orders,
count(distinct case when o.order_status='canceled' then o.order_id end) as canceled_orders
from order_items as oi join orders as o
on oi.order_id=o.order_id
group by oi.seller_id)

select seller_id, total_orders, canceled_orders, round((canceled_orders*100/total_orders),2) as cancellation_rate
from seller_orders
where total_orders>=10
order by cancellation_rate desc
limit 10"""

df=pd.read_sql_query(query,conn)
df

18)Find the peak ordering hours during the day 

In [ ]:
query="""select count(order_id) as total_orders, strftime('%H', order_purchase_timestamp) as hour
from orders
group by hour
order by total_orders desc"""

df=pd.read_sql_query(query,conn)
df

19)Which payment method is most popular per state?

In [ ]:
query="""with payment_count as
(select c.customer_state, p.payment_type, count(*) as total,
rank() over(partition by c.customer_state 
order by count(*) desc) as rnk
from customers as c join orders as o
on c.customer_id=o.customer_id
join payments as p
on p.order_id=o.order_id
group by c.customer_state, p.payment_type)

select customer_state, payment_type, total
from payment_count
where rnk=1
order by customer_state
"""

df=pd.read_sql_query(query,conn)
df

20)Find states with the highest average order value

In [ ]:
query="""select c.customer_state as State, round(avg(p.payment_value),2) as Average_Order_Value
from customers as c join orders as o
on c.customer_id=o.customer_id
join payments as p
on p.order_id=o.order_id
group by State
order by Average_Order_Value desc
"""

df=pd.read_sql_query(query,conn)
df